In [ ]:
# Download your Kaggle .json file and upload it when prompted
# Make sure you downloaded a NEW kaggle.json from kaggle.com/settings/api first

import os, shutil, json
from google.colab import files

# Remove any old broken credentials first
if os.path.exists('/root/.kaggle/kaggle.json'):
    os.remove('/root/.kaggle/kaggle.json')
    print('Removed old kaggle.json')

print('Upload your NEW kaggle.json file:')
uploaded = files.upload()

# Save to correct location
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# Read and print username to confirm it loaded correctly
with open('/root/.kaggle/kaggle.json') as f:
    creds = json.load(f)
print(f'\nCredentials loaded for user: {creds["username"]}')
print('Key starts with:', creds['key'][:6], '...')
print('\nIf your username looks correct, proceed to Fix Cell 2.')

In [ ]:
# This downloads directly into Colab cloud storage.
# Expected time: 15–40 minutes (23.6 GB). Watch for the progress bar.

import os
os.makedirs('/content/asvspoof2019', exist_ok=True)

print('Downloading ASVspoof 2019 dataset...')
print('Do NOT close this browser tab.\n')

!kaggle datasets download -d awsaf49/asvpoof-2019-dataset \
    --unzip -p /content/asvspoof2019

print('\n--- Checking what downloaded ---')
!ls -lh /content/asvspoof2019/

# Confirm the LA folder is present
if os.path.exists('/content/asvspoof2019/LA'):
    print('\n LA folder found. Download successful!')
    print('Go back to Week 1 notebook and run from Cell 4 onwards.')
else:
    print('\n LA folder not found. Check the listing above to see what downloaded.')
    !find /content/asvspoof2019 -maxdepth 2 -type d

In [ ]:
# DIAGNOSTIC — Inspect the downloaded ASVspoof dataset

from pathlib import Path

BASE = Path('/content/asvspoof2019')

print("Top-level contents:")
for item in BASE.iterdir():
    print(" ", item)

print("\nDirectories under /content/asvspoof2019:")
for d in BASE.rglob('*'):
    if d.is_dir():
        print(" ", d)

print("\nSearching for .txt protocol files:")
txt_files = list(BASE.rglob('*.txt'))

if txt_files:
    for f in txt_files[:100]:
        print(" ", f)
else:
    print("❌ No .txt files found anywhere in the downloaded dataset.")

print("\nSearching for evaluation audio:")
flac_dirs = list(BASE.rglob('flac'))

for d in flac_dirs:
    print(" ", d)

In [ ]:
# CELL 4 — Explore the folder structure
# We only need the LA partition (TTS and voice conversion fakes).
# PA contains replay attacks which are not relevant to our project.
from pathlib import Path

BASE_DIR = Path('/content/asvspoof2019')

print('Top-level folders in dataset:')
for folder in sorted(BASE_DIR.iterdir()):
    print(f'  {folder.name}/')

LA_DIR = BASE_DIR / 'LA'
print(f'\nLA directory contents:')
for folder in sorted(LA_DIR.iterdir()):
    print(f'  {folder.name}/')

In [ ]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path('/content/asvspoof2019')

# 🔍 Automatically locate protocol folder
proto_dirs = list(BASE_DIR.rglob("*cm_protocols"))

if not proto_dirs:
    raise FileNotFoundError("Could not find cm_protocols folder anywhere!")

PROTO_DIR = proto_dirs[0]
print("Using protocol directory:", PROTO_DIR)

# 🔍 List actual files inside (VERY IMPORTANT DEBUG STEP)
print("\nFiles inside protocol folder:")
for f in sorted(PROTO_DIR.iterdir()):
    print(f.name)

# Try to auto-detect correct filenames
train_proto = list(PROTO_DIR.glob("*train*.txt"))[0]
dev_proto   = list(PROTO_DIR.glob("*dev*.txt"))[0]
eval_proto  = list(PROTO_DIR.glob("*eval*.txt"))[0]

print("\nDetected files:")
print("Train:", train_proto.name)
print("Dev:", dev_proto.name)
print("Eval:", eval_proto.name)

COL_NAMES = ['speaker_id', 'filename', 'environment', 'attack', 'label']

df_train = pd.read_csv(train_proto, sep=' ', header=None, names=COL_NAMES)
df_dev   = pd.read_csv(dev_proto,   sep=' ', header=None, names=COL_NAMES)
df_eval  = pd.read_csv(eval_proto,  sep=' ', header=None, names=COL_NAMES)

print("\n=== DATASET OVERVIEW ===\n")
print(f"Training: {len(df_train):,}")
print(f"Dev:      {len(df_dev):,}")
print(f"Eval:     {len(df_eval):,}")

In [ ]:
# CELL 6 — Visualise the class distribution
# Save this chart — it belongs in your IEEE paper methodology section.
# Note the heavy imbalance (far more fakes than real) — we handle this in Week 2.

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('ASVspoof 2019 LA — Class Distribution per Split', fontsize=13)

for ax, (df, split) in zip(axes, [(df_train,'Train'),(df_dev,'Dev'),(df_eval,'Eval')]):
    counts = df['label'].value_counts()
    bars = ax.bar(counts.index, counts.values,
                  color=['#2ecc71','#e74c3c'], edgecolor='black', linewidth=0.5)
    ax.set_title(split)
    ax.set_ylabel('Number of files')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{val:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('/content/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to /content/class_distribution.png')

In [ ]:
# CELL 7 — Listen to a real and a fake audio sample
# Listen carefully — the fake may sound slightly robotic or have subtle
# tonal artifacts described in Document 2 (Section 2.2).

import IPython.display as ipd
import librosa

LA_DIR = Path('/content/asvspoof2019/LA/LA')
TRAIN_AUDIO_DIR = LA_DIR / 'ASVspoof2019_LA_train' / 'flac'

real_file = df_train[df_train['label'] == 'bonafide']['filename'].iloc[0]
fake_file = df_train[df_train['label'] == 'spoof']['filename'].iloc[0]

real_path = TRAIN_AUDIO_DIR / f'{real_file}.flac'
fake_path = TRAIN_AUDIO_DIR / f'{fake_file}.flac'

print(f'Playing REAL audio sample: {real_file}')
audio_real, sr = librosa.load(real_path, sr=16000)
ipd.display(ipd.Audio(audio_real, rate=sr))

print(f'\nPlaying FAKE (cloned) audio sample: {fake_file}')
audio_fake, sr = librosa.load(fake_path, sr=16000)
ipd.display(ipd.Audio(audio_fake, rate=sr))

print(f'\nSample rate: {sr} Hz')
print(f'Real clip duration: {len(audio_real)/sr:.2f} seconds')
print(f'Fake clip duration: {len(audio_fake)/sr:.2f} seconds')

In [ ]:
# CELL 8 — Visualise spectrograms of real vs fake
# X axis = time, Y axis = frequency, Colour = energy
# Differences between real and fake are subtle — this is why we need a CNN.
# Save this figure — it belongs in your IEEE paper methodology section.
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Mel Spectrogram: Real vs Fake Voice', fontsize=13)

for ax, (audio, title) in zip(axes, [(audio_real,'REAL (bonafide)'),
                                      (audio_fake,'FAKE (spoofed)')]):
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_mels=64, fmax=8000)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, sr=sr, x_axis='time',
                                   y_axis='mel', ax=ax, fmax=8000)
    ax.set_title(title, fontsize=11)
    fig.colorbar(img, ax=ax, format='%+2.0f dB')

plt.tight_layout()
plt.savefig('/content/spectrogram_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Spectrogram comparison saved to /content/spectrogram_comparison.png')

In [ ]:
# CELL 9 — Define the feature extraction function
# Converts one audio file into a fixed-size (64, 128) Mel spectrogram tensor.
# Short clips are padded. Long clips are truncated. All values normalised to [0,1].

def extract_mel_spectrogram(file_path, sr=16000, n_mels=64, n_frames=128, fmax=8000):
    """
    Load one audio file and return a fixed-size Mel spectrogram.
    Returns numpy array of shape (64, 128), values in range [0, 1].
    """
    try:
        audio, _ = librosa.load(file_path, sr=sr, mono=True)

        mel = librosa.feature.melspectrogram(
            y=audio, sr=sr, n_mels=n_mels, fmax=fmax,
            hop_length=512, n_fft=1024
        )
        mel_db = librosa.power_to_db(mel, ref=np.max)

        # Fix time dimension to exactly n_frames
        if mel_db.shape[1] < n_frames:
            pad_width = n_frames - mel_db.shape[1]
            mel_db = np.pad(mel_db, ((0,0),(0,pad_width)),
                           mode='constant', constant_values=mel_db.min())
        else:
            mel_db = mel_db[:, :n_frames]

        # Normalise to [0, 1]
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
        return mel_db

    except Exception as e:
        print(f'Error processing {file_path}: {e}')
        return None


# Test on one file
test_feature = extract_mel_spectrogram(real_path)
print(f'Feature shape: {test_feature.shape}')   # should be (64, 128)
print(f'Value range:   [{test_feature.min():.3f}, {test_feature.max():.3f}]')
print('Feature extraction function working correctly.')

In [ ]:
# CELL 10 — Extract features from the full training set
# Loops through all training audio files and extracts spectrograms.
# Expected time: 20-40 minutes. Progress bar shows how many files are done.

from tqdm import tqdm
def extract_features_for_split(df, audio_dir, split_name):
    features, labels, failed = [], [], []
    print(f'\nExtracting features for {split_name} set ({len(df)} files)...')

    for _, row in tqdm(df.iterrows(), total=len(df), desc=split_name):
        file_path = audio_dir / f"{row['filename']}.flac"

        if not file_path.exists():
            failed.append(row['filename'])
            continue

        mel = extract_mel_spectrogram(file_path)
        if mel is not None:
            features.append(mel)
            labels.append(0 if row['label'] == 'bonafide' else 1)
        else:
            failed.append(row['filename'])

    features = np.array(features, dtype=np.float32)
    labels   = np.array(labels,   dtype=np.int32)

    print(f'  Done. Extracted {len(features):,} features.')
    print(f'  Real (0): {(labels==0).sum():,}  |  Fake (1): {(labels==1).sum():,}')
    if failed:
        print(f'  Failed: {len(failed)} files skipped.')

    return features, labels, failed


TRAIN_AUDIO = LA_DIR / 'ASVspoof2019_LA_train' / 'flac'
X_train, y_train, failed_train = extract_features_for_split(df_train, TRAIN_AUDIO, 'Training')

print(f'\nTraining features shape: {X_train.shape}')   # (N, 64, 128)
print(f'Training labels shape:   {y_train.shape}')

In [ ]:
# CELL 11 — Extract features from development and evaluation sets
# These are validation and test sets — we NEVER train on these.
# Keeping them separate gives honest accuracy numbers for IEEE.

DEV_AUDIO  = LA_DIR / 'ASVspoof2019_LA_dev'  / 'flac'
EVAL_AUDIO = LA_DIR / 'ASVspoof2019_LA_eval' / 'flac'

X_dev,  y_dev,  failed_dev  = extract_features_for_split(df_dev,  DEV_AUDIO,  'Development')
X_eval, y_eval, failed_eval = extract_features_for_split(df_eval, EVAL_AUDIO, 'Evaluation')

print('\n=== FINAL FEATURE SUMMARY ===')
print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_dev:   {X_dev.shape}    y_dev:   {y_dev.shape}')
print(f'X_eval:  {X_eval.shape}   y_eval:  {y_eval.shape}')

In [ ]:
# CELL 12 — Save all features to disk as .npy files
# .npy is numpy's native format — fast to save and load.
# Saving now means Week 2 starts immediately without re-extracting.

SAVE_DIR = Path('/content/features')
SAVE_DIR.mkdir(exist_ok=True)

np.save(SAVE_DIR / 'X_train.npy', X_train)
np.save(SAVE_DIR / 'y_train.npy', y_train)
np.save(SAVE_DIR / 'X_dev.npy',   X_dev)
np.save(SAVE_DIR / 'y_dev.npy',   y_dev)
np.save(SAVE_DIR / 'X_eval.npy',  X_eval)
np.save(SAVE_DIR / 'y_eval.npy',  y_eval)

print('All features saved to /content/features/')
print('\nFiles saved:')
for f in sorted(SAVE_DIR.iterdir()):
    size_mb = f.stat().st_size / (1024**2)
    print(f'  {f.name:20s}  {size_mb:.1f} MB')

In [ ]:
# CELL 13 — Save feature files to Google Drive (IMPORTANT)
# Colab sessions reset after a few hours — /content/ disappears.
# This cell copies everything to Drive so it persists permanently.
# You will be asked to authorise Google Drive access — click the link and paste the code.

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/ICAST_Project/features')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

for f in SAVE_DIR.iterdir():
    shutil.copy(f, DRIVE_DIR / f.name)
    print(f'Copied {f.name} to Google Drive')

# Also save the charts
CHARTS_DIR = Path('/content/drive/MyDrive/ICAST_Project/charts')
CHARTS_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy('/content/class_distribution.png',    CHARTS_DIR / 'class_distribution.png')
shutil.copy('/content/spectrogram_comparison.png', CHARTS_DIR / 'spectrogram_comparison.png')

print('\nAll features and charts saved to Google Drive.')
print('Your files are safe even if this Colab session ends.')

In [ ]:
# CELL 14 — Final verification and Week 1 summary
# Reloads features from Drive to confirm everything saved correctly.

print('=' * 60)
print('WEEK 1 COMPLETE — VERIFICATION SUMMARY')
print('=' * 60)

X_train_check = np.load(DRIVE_DIR / 'X_train.npy')
y_train_check = np.load(DRIVE_DIR / 'y_train.npy')

print(f'\nLoaded X_train from Drive: {X_train_check.shape}')
print(f'Loaded y_train from Drive: {y_train_check.shape}')
print(f'Real={(y_train_check==0).sum():,}  Fake={(y_train_check==1).sum():,}')

assert X_train_check.shape[1] == 64,  'ERROR: Wrong number of mel bands'
assert X_train_check.shape[2] == 128, 'ERROR: Wrong number of time frames'
assert X_train_check.dtype == np.float32, 'ERROR: Wrong data type'

print('\nAll checks passed.')
print('\n--- Ready for Week 2 ---')
print('In Week 2 we will:')
print('  1. Build the MFCC-GMM baseline model')
print('  2. Build and train the lightweight CNN')
print('  3. Compare accuracy and EER between both models')
print('  4. Generate your first results table for the IEEE paper')
print('=' * 60)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
#Checking the contents of my drive to see the exact location


from pathlib import Path
DRIVE = Path('/content/drive/MyDrive')
print("Contents of MyDrive:")
for f in sorted(DRIVE.iterdir()):
    print(f"  {f.name}")



from pathlib import Path

ICAST_DIR = Path('/content/drive/MyDrive/ICAST_Project')
print("Contents of ICAST_Project:")
for f in sorted(ICAST_DIR.rglob('*')):
    print(f"  {f.relative_to(ICAST_DIR)}")

In [ ]:
import numpy as np
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/ICAST_Project/features')
print("Loading features...")
X_train = np.load(DRIVE_DIR / 'X_train.npy')
y_train = np.load(DRIVE_DIR / 'y_train.npy')
X_dev   = np.load(DRIVE_DIR / 'X_dev.npy')
y_dev   = np.load(DRIVE_DIR / 'y_dev.npy')
X_eval  = np.load(DRIVE_DIR / 'X_eval.npy')
y_eval  = np.load(DRIVE_DIR / 'y_eval.npy')

print(f"X_train: {X_train.shape}")
print(f"X_dev:   {X_dev.shape}")
print(f"X_eval:  {X_eval.shape}")
print("\nAll features loaded. Ready for Week 2.")

In [ ]:
# BRIDGING CELL
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import time, shutil, os, warnings, pickle, gc
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PROJECT_DIR = Path('/content/drive/MyDrive/ICAST_Project')
DRIVE_DIR   = Path('/content/drive/MyDrive/ICAST_Project/features')

print(f"Device: {device}")
print("All imports ready. Proceed to Cell 3.")

In [ ]:
# CELL 2 — Memory-efficient Dataset class
# Instead of loading ALL 3.8 GB into RAM at once, this class loads
# only one small chunk at a time during training.
# RAM usage drops from ~4 GB to ~200 MB.

class ChunkedNpyDataset(Dataset):
    """
    Loads .npy features from disk in chunks to save RAM.
    Each chunk is loaded, used, then discarded.
    """
    def __init__(self, X_path, y_path, chunk_size=2000):
        self.X_path     = X_path
        self.y_path     = y_path
        self.chunk_size = chunk_size

        # Load only labels into RAM (tiny — 0.1 MB)
        self.y = np.load(y_path)
        self.n = len(self.y)

        # Load X in a memory-mapped mode — reads from disk on demand
        self.X = np.load(X_path, mmap_mode='r')

        print(f'Dataset ready: {self.n:,} samples  (memory-mapped, minimal RAM)')

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        # Read one sample from disk — shape (64, 128) -> (1, 64, 128)
        x = torch.FloatTensor(self.X[idx]).unsqueeze(0)
        y = torch.FloatTensor([self.y[idx]])
        return x, y

    def get_labels(self):
        return self.y


# Create datasets for all three splits
print('Creating memory-efficient datasets...')
train_dataset = ChunkedNpyDataset(DRIVE_DIR/'X_train.npy', DRIVE_DIR/'y_train.npy')
dev_dataset   = ChunkedNpyDataset(DRIVE_DIR/'X_dev.npy',   DRIVE_DIR/'y_dev.npy')
eval_dataset  = ChunkedNpyDataset(DRIVE_DIR/'X_eval.npy',  DRIVE_DIR/'y_eval.npy')

# Labels for class weighting
y_train = train_dataset.get_labels()
y_dev   = dev_dataset.get_labels()
y_eval  = eval_dataset.get_labels()

print(f'\nTrain: {len(train_dataset):,}  Real: {(y_train==0).sum():,}  Fake: {(y_train==1).sum():,}')
print(f'Dev:   {len(dev_dataset):,}  Real: {(y_dev==0).sum():,}  Fake: {(y_dev==1).sum():,}')
print(f'Eval:  {len(eval_dataset):,}  Real: {(y_eval==0).sum():,}  Fake: {(y_eval==1).sum():,}')

In [ ]:
# CELL 3 — Create DataLoaders with weighted sampling
# num_workers=2 allows parallel data loading from disk
# pin_memory=True speeds up GPU transfer

# Weighted sampler to handle class imbalance
class_counts  = np.bincount(y_train.astype(int))
class_weights = 1.0 / class_counts
sample_weights = class_weights[y_train.astype(int)]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(
    train_dataset, batch_size=64, sampler=sampler,
    num_workers=2, pin_memory=True
)
dev_loader = DataLoader(
    dev_dataset, batch_size=64, shuffle=False,
    num_workers=2, pin_memory=True
)
eval_loader = DataLoader(
    eval_dataset, batch_size=64, shuffle=False,
    num_workers=2, pin_memory=True
)

print(f'Train batches:      {len(train_loader)}')
print(f'Validation batches: {len(dev_loader)}')
print(f'Test batches:       {len(eval_loader)}')
print('DataLoaders ready.')

In [ ]:
# CELL 4 — Helper metrics functions (EER, evaluation summary)

def compute_eer(y_true, y_scores):
    fpr, tpr, _ = roc_curve(y_true, y_scores, pos_label=1)
    fnr = 1 - tpr
    eer_idx = np.argmin(np.abs(fpr - fnr))
    return round((fpr[eer_idx] + fnr[eer_idx]) / 2 * 100, 2)


def evaluate_model_metrics(y_true, y_pred, y_scores, model_name, inference_ms, size_mb):
    acc = accuracy_score(y_true, y_pred) * 100
    eer = compute_eer(y_true, y_scores)
    cm  = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    far = fp / (fp + tn) * 100
    frr = fn / (fn + tp) * 100

    print(f'\n=== {model_name} ===')
    print(f'  Accuracy:          {acc:.2f}%')
    print(f'  EER:               {eer:.2f}%')
    print(f'  False Accept Rate: {far:.2f}%')
    print(f'  False Reject Rate: {frr:.2f}%')
    print(f'  Model size:        {size_mb:.2f} MB')
    print(f'  Inference time:    {inference_ms:.1f} ms/sample')

    return {
        'Model': model_name,
        'Accuracy (%)': round(acc, 2),
        'EER (%)': eer,
        'FAR (%)': round(far, 2),
        'FRR (%)': round(frr, 2),
        'Size (MB)': round(size_mb, 2),
        'Inference (ms)': round(inference_ms, 1)
    }

print('Helper functions defined.')

In [ ]:
# CELL 5 — MFCC-GMM Baseline
# Uses a small random sample (5000 files) for GMM fitting to save RAM.
# This is standard practice for GMM on large datasets.

print('Training MFCC-GMM baseline (memory-efficient version)...')

# Load only a subset for GMM (5000 samples fits comfortably in RAM)
SAMPLE_SIZE = 5000
np.random.seed(42)

X_full = np.load(DRIVE_DIR/'X_train.npy', mmap_mode='r')
real_idx = np.where(y_train == 0)[0]
fake_idx = np.where(y_train == 1)[0]

# Sample equally from real and fake
sample_real = np.random.choice(real_idx, min(SAMPLE_SIZE//2, len(real_idx)), replace=False)
sample_fake = np.random.choice(fake_idx, min(SAMPLE_SIZE//2, len(fake_idx)), replace=False)

X_gmm_real = X_full[sample_real].reshape(len(sample_real), -1).copy()
X_gmm_fake = X_full[sample_fake].reshape(len(sample_fake), -1).copy()
del X_full; gc.collect()

scaler = StandardScaler()
X_gmm_real_s = scaler.fit_transform(X_gmm_real)
X_gmm_fake_s = scaler.transform(X_gmm_fake)

print('Fitting GMM for REAL class...')
gmm_real = GaussianMixture(n_components=8, covariance_type='diag', max_iter=100, random_state=42)
gmm_real.fit(X_gmm_real_s)

print('Fitting GMM for FAKE class...')
gmm_fake = GaussianMixture(n_components=8, covariance_type='diag', max_iter=100, random_state=42)
gmm_fake.fit(X_gmm_fake_s)

del X_gmm_real, X_gmm_fake, X_gmm_real_s, X_gmm_fake_s; gc.collect()

# Evaluate on eval set in chunks to save RAM
print('\nEvaluating GMM on evaluation set in chunks...')
X_eval_full = np.load(DRIVE_DIR/'X_eval.npy', mmap_mode='r')
CHUNK = 1000
gmm_pred, gmm_scores_list = [], []
start = time.time()

for i in range(0, len(y_eval), CHUNK):
    chunk = X_eval_full[i:i+CHUNK].reshape(len(X_eval_full[i:i+CHUNK]), -1).copy()
    chunk_s = scaler.transform(chunk)
    s_real  = gmm_real.score_samples(chunk_s)
    s_fake  = gmm_fake.score_samples(chunk_s)
    gmm_pred.extend((s_fake > s_real).astype(int))
    gmm_scores_list.extend(s_fake - s_real)
    del chunk, chunk_s

gmm_inference_ms = (time.time()-start) / len(y_eval) * 1000
gmm_pred   = np.array(gmm_pred)
gmm_scores = np.array(gmm_scores_list)
gmm_scores = (gmm_scores-gmm_scores.min())/(gmm_scores.max()-gmm_scores.min())
del X_eval_full; gc.collect()

gmm_path = '/content/gmm_model.pkl'
with open(gmm_path, 'wb') as f:
    pickle.dump({'gmm_real':gmm_real,'gmm_fake':gmm_fake,'scaler':scaler}, f)
gmm_size_mb = os.path.getsize(gmm_path)/(1024**2)

gmm_results = evaluate_model_metrics(
    y_eval, gmm_pred, gmm_scores, 'MFCC-GMM Baseline', gmm_inference_ms, gmm_size_mb
)
print('\nGMM baseline complete.')

In [ ]:
# CELL 6 — CNN Model definitions

class LightweightCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2,2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(64*8*16, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.classifier(self.features(x))


class FullCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2,2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(128*8*16, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 1), nn.Sigmoid()
        )
    def forward(self, x):
        return self.classifier(self.features(x))


light_params = sum(p.numel() for p in LightweightCNN().parameters())
full_params  = sum(p.numel() for p in FullCNN().parameters())
print(f'LightweightCNN: {light_params:,} parameters  (~{light_params*4/1e6:.2f} MB)')
print(f'FullCNN:        {full_params:,} parameters  (~{full_params*4/1e6:.2f} MB)')
print(f'Size reduction: {full_params/light_params:.1f}x')

In [ ]:
# CELL 7 — Training function with memory cleanup

def train_model(model, train_loader, dev_loader, model_name,
                max_epochs=25, patience=5, lr=0.001):
    model = model.to(device)
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

    history = {'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[]}
    best_val_loss = float('inf')
    best_path = f'/content/{model_name}_best.pth'
    no_improve = 0

    print(f'\nTraining {model_name}...')
    print('-'*65)
    print(f'{"Epoch":>6} {"Train Loss":>12} {"Val Loss":>10} {"Train Acc":>11} {"Val Acc":>9}')
    print('-'*65)

    for epoch in range(1, max_epochs+1):
        # Training
        model.train()
        t_loss, t_correct, t_total = 0, 0, 0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            out  = model(X_b)
            loss = criterion(out, y_b)
            loss.backward()
            optimizer.step()
            t_loss    += loss.item() * len(y_b)
            t_correct += ((out>=0.5).float()==y_b).sum().item()
            t_total   += len(y_b)

        # Validation
        model.eval()
        v_loss, v_correct, v_total = 0, 0, 0
        with torch.no_grad():
            for X_b, y_b in dev_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                out  = model(X_b)
                loss = criterion(out, y_b)
                v_loss    += loss.item() * len(y_b)
                v_correct += ((out>=0.5).float()==y_b).sum().item()
                v_total   += len(y_b)

        tl = t_loss/t_total; vl = v_loss/v_total
        ta = t_correct/t_total*100; va = v_correct/v_total*100
        history['train_loss'].append(tl); history['val_loss'].append(vl)
        history['train_acc'].append(ta);  history['val_acc'].append(va)
        scheduler.step(vl)
        print(f'{epoch:>6} {tl:>12.4f} {vl:>10.4f} {ta:>10.2f}% {va:>8.2f}%')

        if vl < best_val_loss:
            best_val_loss = vl
            torch.save(model.state_dict(), best_path)
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stopping at epoch {epoch}.')
                break

        # Free GPU cache each epoch
        torch.cuda.empty_cache(); gc.collect()

    print(f'Best val loss: {best_val_loss:.4f} — saved to {best_path}')
    return history, best_path

print('Training function defined.')

In [ ]:
# CELL 8 — Train LightweightCNN
# Expected time: 15-25 minutes on T4 GPU

light_model = LightweightCNN()
light_history, light_best_path = train_model(
    light_model, train_loader, dev_loader,
    model_name='LightweightCNN', max_epochs=25, patience=5
)
del light_model; torch.cuda.empty_cache(); gc.collect()
print('LightweightCNN training complete.')

In [ ]:
# CELL 9 — Train FullCNN
# Expected time: 25-40 minutes on T4 GPU

full_model = FullCNN()
full_history, full_best_path = train_model(
    full_model, train_loader, dev_loader,
    model_name='FullCNN', max_epochs=25, patience=5
)
del full_model; torch.cuda.empty_cache(); gc.collect()
print('FullCNN training complete.')

In [ ]:
# CELL 11 — Evaluate both CNNs on evaluation set

def evaluate_cnn(ModelClass, best_path, eval_loader, model_name):
    model = ModelClass().to(device)
    model.load_state_dict(torch.load(best_path, map_location=device))
    model.eval()
    all_preds, all_scores = [], []
    start = time.time()
    with torch.no_grad():
        for X_b, _ in eval_loader:
            out = model(X_b.to(device)).squeeze(1)
            all_preds.extend((out>=0.5).long().cpu().numpy())
            all_scores.extend(out.cpu().numpy())
    inf_ms  = (time.time()-start)/len(eval_loader.dataset)*1000
    size_mb = os.path.getsize(best_path)/(1024**2)
    del model; torch.cuda.empty_cache(); gc.collect()
    return np.array(all_preds), np.array(all_scores), inf_ms, size_mb


print('Evaluating LightweightCNN...')
light_pred, light_scores, light_time, light_size = evaluate_cnn(
    LightweightCNN, light_best_path, eval_loader, 'LightweightCNN')
light_results = evaluate_model_metrics(
    y_eval, light_pred, light_scores, 'Lightweight CNN', light_time, light_size)

print('\nEvaluating FullCNN...')
full_pred, full_scores, full_time, full_size = evaluate_cnn(
    FullCNN, full_best_path, eval_loader, 'FullCNN')
full_results = evaluate_model_metrics(
    y_eval, full_pred, full_scores, 'Full CNN', full_time, full_size)

In [ ]:
# CELL 12 — IEEE Results Table 1

results_df = pd.DataFrame([gmm_results, light_results, full_results]).set_index('Model')

print('\n' + '='*75)
print('TABLE 1: MODEL COMPARISON ON ASVspoof 2019 LA EVALUATION SET (CLEAN AUDIO)')
print('='*75)
print(results_df.to_string())
print('='*75)
print('EER=Equal Error Rate | FAR=False Accept Rate | FRR=False Reject Rate')

results_df.to_csv('/content/results_table1_clean_audio.csv')

acc_drop   = full_results['Accuracy (%)'] - light_results['Accuracy (%)']
size_ratio = full_results['Size (MB)'] / light_results['Size (MB)']
print(f'\nKEY FINDING: LightweightCNN is {size_ratio:.1f}x smaller than FullCNN')
print(f'with only {abs(acc_drop):.2f}% accuracy difference — viable for mobile deployment.')

In [ ]:
# CELL 13 — Confusion matrices

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Confusion Matrices — All Models on ASVspoof 2019 LA Evaluation Set', fontsize=13)

for ax, (pred, name) in zip(axes, [
    (gmm_pred,'MFCC-GMM Baseline'),(light_pred,'Lightweight CNN'),(full_pred,'Full CNN')
]):
    cm = confusion_matrix(y_eval, pred)
    ax.imshow(cm, cmap=plt.cm.Blues)
    ax.set_title(name, fontsize=11)
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['Real','Fake']); ax.set_yticklabels(['Real','Fake'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                   color='white' if cm[i,j]>cm.max()/2 else 'black',
                   fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrices saved.')

In [ ]:
# CELL 14 — Save everything to Google Drive

MODELS_DIR  = PROJECT_DIR / 'models';  MODELS_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR  = PROJECT_DIR / 'charts';  CHARTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = PROJECT_DIR / 'results'; RESULTS_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy(light_best_path,             MODELS_DIR/'LightweightCNN_best.pth')
shutil.copy(full_best_path,              MODELS_DIR/'FullCNN_best.pth')
shutil.copy('/content/gmm_model.pkl',    MODELS_DIR/'gmm_model.pkl')
shutil.copy('/content/training_curves.png',   CHARTS_DIR/'training_curves.png')
shutil.copy('/content/confusion_matrices.png', CHARTS_DIR/'confusion_matrices.png')
shutil.copy('/content/results_table1_clean_audio.csv', RESULTS_DIR/'results_table1_clean_audio.csv')

print('All models, charts and results saved to Google Drive.')
print('\n=== WEEK 2 COMPLETE ===')
print('Ready for Week 3: robustness testing + quantization')

In [ ]:
# CELL 1 — Reinstall libraries and reload everything from Drive


!pip install torch torchaudio scikit-learn --quiet

import numpy as np
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import time, shutil, os, warnings, pickle, gc, io
import librosa
import scipy.signal
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

device      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DRIVE_DIR   = Path('/content/drive/MyDrive/ICAST_Project/features')
PROJECT_DIR = Path('/content/drive/MyDrive/ICAST_Project')
MODELS_DIR  = PROJECT_DIR / 'models'
CHARTS_DIR  = PROJECT_DIR / 'charts'
RESULTS_DIR = PROJECT_DIR / 'results'
for d in [CHARTS_DIR, RESULTS_DIR]: d.mkdir(parents=True, exist_ok=True)

print(f'Device: {device}')
print('All libraries loaded.')

In [ ]:
# CELL 2 — Redefine model architectures and load trained weights
# We must redefine the classes before loading the saved weights.

class LightweightCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2,2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(64*8*16, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.classifier(self.features(x))


class FullCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2,2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(128*8*16, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.classifier(self.features(x))


# Load trained weights
light_model = LightweightCNN()
light_model.load_state_dict(torch.load(MODELS_DIR/'LightweightCNN_best.pth', map_location=device))
light_model = light_model.to(device)
light_model.eval()

full_model = FullCNN()
full_model.load_state_dict(torch.load(MODELS_DIR/'FullCNN_best.pth', map_location=device))
full_model = full_model.to(device)
full_model.eval()

# Load GMM
with open(MODELS_DIR/'gmm_model.pkl', 'rb') as f:
    gmm_data = pickle.load(f)
gmm_real = gmm_data['gmm_real']
gmm_fake = gmm_data['gmm_fake']
scaler   = gmm_data['scaler']

# Load labels
y_eval = np.load(DRIVE_DIR/'y_eval.npy')

print('All models loaded successfully.')
print(f'LightweightCNN: {sum(p.numel() for p in light_model.parameters()):,} parameters')
print(f'FullCNN:        {sum(p.numel() for p in full_model.parameters()):,} parameters')

In [ ]:
# CELL 3 — Define telephone channel distortion functions
# This is the core of Week 3 and our key research contribution.
# We simulate three types of distortion common in phone banking calls:
#
# 1. G.711 codec simulation — the standard codec used in telephone networks.
#    It compresses audio to 8kHz (from 16kHz) then upsamples back, causing
#    high-frequency detail loss. This is what phone calls actually sound like.
#
# 2. Background noise — simulates ambient noise in a real call environment
#    (office noise, street noise, etc.) at a realistic SNR of 20dB.
#
# 3. Combined — both distortions applied together, the most realistic scenario.
#
# Document 2 Section 4.1 identifies these as the primary causes of domain shift
# — the main reason detection models fail in real-world deployment.

def simulate_g711_codec(spectrogram, sr=16000):
    """
    Simulate G.711 telephone codec on a mel spectrogram.
    G.711 operates at 8kHz, so we lose all frequencies above 4kHz.
    We simulate this by zeroing out the upper mel bands.
    """
    distorted = spectrogram.copy()
    # G.711 cuts frequencies above 4kHz — roughly upper 40% of mel bands
    cutoff_band = int(distorted.shape[0] * 0.6)
    distorted[cutoff_band:, :] = 0.0
    # Add mild quantization noise (8-bit codec introduces quantization error)
    quantization_noise = np.random.uniform(-0.02, 0.02, distorted.shape)
    distorted = np.clip(distorted + quantization_noise, 0, 1)
    return distorted


def add_background_noise(spectrogram, snr_db=20):
    """
    Add white Gaussian noise at a given SNR (Signal-to-Noise Ratio).
    SNR of 20dB represents mild background noise (quiet office).
    SNR of 10dB represents moderate noise (busy street).
    """
    signal_power = np.mean(spectrogram ** 2)
    noise_power  = signal_power / (10 ** (snr_db / 10))
    noise = np.random.normal(0, np.sqrt(noise_power), spectrogram.shape)
    return np.clip(spectrogram + noise, 0, 1)


def apply_combined_distortion(spectrogram):
    """
    Apply both codec simulation and background noise.
    This is the most realistic phone banking scenario.
    """
    distorted = simulate_g711_codec(spectrogram)
    distorted = add_background_noise(distorted, snr_db=20)
    return distorted


print('Distortion functions defined.')
print('Three distortion types:')
print('  1. G.711 codec simulation (telephone bandwidth limitation)')
print('  2. Background noise at 20dB SNR')
print('  3. Combined (codec + noise) — most realistic')

In [ ]:
# CELL 4 — Visualise the effect of distortion
# Shows what happens to a spectrogram after each type of distortion.
# For IEEE paper: this figure goes in your methodology section.

# Load one sample spectrogram for visualisation
X_sample = np.load(DRIVE_DIR/'X_eval.npy', mmap_mode='r')[0].copy()

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Effect of Telephone Channel Distortion on Mel Spectrogram', fontsize=13)

titles = ['Original (Clean)', 'G.711 Codec', 'Background Noise (20dB)', 'Combined']
specs  = [
    X_sample,
    simulate_g711_codec(X_sample),
    add_background_noise(X_sample),
    apply_combined_distortion(X_sample)
]

for ax, spec, title in zip(axes, specs, titles):
    im = ax.imshow(spec, aspect='auto', origin='lower', cmap='viridis')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Time frames')
    ax.set_ylabel('Mel bands')
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('/content/distortion_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Distortion visualisation saved.')

In [ ]:
# CELL 5 — Define distorted dataset and evaluation function
# Applies distortion on-the-fly during evaluation to save RAM.
# No need to save distorted versions to disk.

class DistortedDataset(Dataset):
    """
    Loads spectrograms and applies distortion on-the-fly.
    distortion_fn: function to apply to each spectrogram
    """
    def __init__(self, X_path, y_path, distortion_fn=None):
        self.X = np.load(X_path, mmap_mode='r')
        self.y = np.load(y_path)
        self.distortion_fn = distortion_fn

    def __len__(self): return len(self.y)

    def __getitem__(self, idx):
        spec = self.X[idx].copy()
        if self.distortion_fn:
            spec = self.distortion_fn(spec)
        x = torch.FloatTensor(spec).unsqueeze(0)
        y = torch.FloatTensor([self.y[idx]])
        return x, y


def compute_eer(y_true, y_scores):
    fpr, tpr, _ = roc_curve(y_true, y_scores, pos_label=1)
    fnr = 1 - tpr
    idx = np.argmin(np.abs(fpr - fnr))
    return round((fpr[idx] + fnr[idx]) / 2 * 100, 2)


def evaluate_cnn_on_loader(model, loader, model_name, condition):
    """
    Evaluate a CNN model on any DataLoader and return metrics.
    """
    model.eval()
    all_preds, all_scores = [], []
    start = time.time()

    with torch.no_grad():
        for X_b, _ in loader:
            out = model(X_b.to(device)).squeeze(1)
            all_preds.extend((out >= 0.5).long().cpu().numpy())
            all_scores.extend(out.cpu().numpy())

    inf_ms   = (time.time()-start) / len(loader.dataset) * 1000
    preds    = np.array(all_preds)
    scores   = np.array(all_scores)
    acc      = accuracy_score(y_eval, preds) * 100
    eer      = compute_eer(y_eval, scores)
    cm       = confusion_matrix(y_eval, preds)
    tn,fp,fn,tp = cm.ravel()
    far      = fp/(fp+tn)*100
    frr      = fn/(fn+tp)*100
    size_mb  = os.path.getsize(MODELS_DIR/f'{model_name}_best.pth')/(1024**2)

    print(f'  [{condition}] Acc: {acc:.2f}%  EER: {eer:.2f}%  '
          f'FAR: {far:.2f}%  FRR: {frr:.2f}%  '
          f'Size: {size_mb:.2f}MB  Inf: {inf_ms:.1f}ms')

    return {
        'Model': model_name, 'Condition': condition,
        'Accuracy (%)': round(acc,2), 'EER (%)': eer,
        'FAR (%)': round(far,2), 'FRR (%)': round(frr,2),
        'Size (MB)': round(size_mb,2), 'Inference (ms)': round(inf_ms,1)
    }


print('Dataset and evaluation functions defined.')

In [ ]:
# CELL 6 — Evaluate LightweightCNN under all conditions
# Tests the model on clean audio, then three distorted versions.
# Expected time: 5-10 minutes total.

print('=== EVALUATING LIGHTWEIGHT CNN ===')
light_all_results = []

conditions = [
    ('Clean',    None),
    ('G711',     simulate_g711_codec),
    ('Noise',    add_background_noise),
    ('Combined', apply_combined_distortion)
]

for cond_name, distortion_fn in conditions:
    dataset = DistortedDataset(
        DRIVE_DIR/'X_eval.npy',
        DRIVE_DIR/'y_eval.npy',
        distortion_fn=distortion_fn
    )
    loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)
    result = evaluate_cnn_on_loader(light_model, loader, 'LightweightCNN', cond_name)
    light_all_results.append(result)
    del dataset, loader; gc.collect()

print('\nLightweightCNN evaluation complete.')

In [ ]:
# CELL 7 — Evaluate FullCNN under all conditions
# Same four conditions as above for direct comparison.

print('=== EVALUATING FULL CNN ===')
full_all_results = []

for cond_name, distortion_fn in conditions:
    dataset = DistortedDataset(
        DRIVE_DIR/'X_eval.npy',
        DRIVE_DIR/'y_eval.npy',
        distortion_fn=distortion_fn
    )
    loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)
    result = evaluate_cnn_on_loader(full_model, loader, 'FullCNN', cond_name)
    full_all_results.append(result)
    del dataset, loader; gc.collect()

print('\nFullCNN evaluation complete.')

In [ ]:
# CELL 8 — Evaluate GMM under all conditions
# GMM evaluation uses chunked loading to save RAM.

print('=== EVALUATING GMM BASELINE ===')
gmm_all_results = []
CHUNK = 1000

for cond_name, distortion_fn in conditions:
    X_mmap = np.load(DRIVE_DIR/'X_eval.npy', mmap_mode='r')
    gmm_preds, gmm_scores_list = [], []
    start = time.time()

    for i in range(0, len(y_eval), CHUNK):
        chunk = X_mmap[i:i+CHUNK].copy()
        if distortion_fn:
            chunk = np.array([distortion_fn(s) for s in chunk])
        chunk_flat = chunk.reshape(len(chunk), -1)
        chunk_s    = scaler.transform(chunk_flat)
        s_real = gmm_real.score_samples(chunk_s)
        s_fake = gmm_fake.score_samples(chunk_s)
        gmm_preds.extend((s_fake > s_real).astype(int))
        gmm_scores_list.extend(s_fake - s_real)
        del chunk, chunk_flat, chunk_s

    inf_ms   = (time.time()-start) / len(y_eval) * 1000
    preds    = np.array(gmm_preds)
    scores   = np.array(gmm_scores_list)
    scores   = (scores-scores.min())/(scores.max()-scores.min()+1e-8)
    acc      = accuracy_score(y_eval, preds) * 100
    eer      = compute_eer(y_eval, scores)
    cm       = confusion_matrix(y_eval, preds)
    tn,fp,fn,tp = cm.ravel()
    far = fp/(fp+tn)*100; frr = fn/(fn+tp)*100
    size_mb = os.path.getsize(MODELS_DIR/'gmm_model.pkl')/(1024**2)

    print(f'  [{cond_name}] Acc: {acc:.2f}%  EER: {eer:.2f}%  '
          f'FAR: {far:.2f}%  Size: {size_mb:.2f}MB  Inf: {inf_ms:.1f}ms')

    gmm_all_results.append({
        'Model':'GMM-Baseline','Condition':cond_name,
        'Accuracy (%)':round(acc,2),'EER (%)':eer,
        'FAR (%)':round(far,2),'FRR (%)':round(frr,2),
        'Size (MB)':round(size_mb,2),'Inference (ms)':round(inf_ms,1)
    })
    del X_mmap; gc.collect()

print('\nGMM evaluation complete.')

In [ ]:
# CELL 9 — INT8 Quantization of LightweightCNN
# Quantization converts 32-bit float weights to 8-bit integers.
# This makes the model 4x smaller and 2-3x faster on CPU.
# It is the key technique for mobile deployment described in Document 3.
#
# We use PyTorch's built-in dynamic quantization which:
# - Requires no retraining
# - Automatically quantizes Linear layers
# - Preserves accuracy within ~1-2%

print('Applying INT8 dynamic quantization to LightweightCNN...')

# Move model to CPU for quantization (quantization runs on CPU)
light_model_cpu = LightweightCNN()
light_model_cpu.load_state_dict(
    torch.load(MODELS_DIR/'LightweightCNN_best.pth', map_location='cpu')
)
light_model_cpu.eval()

# Apply dynamic INT8 quantization
quantized_model = torch.quantization.quantize_dynamic(
    light_model_cpu,
    {nn.Linear},          # quantize Linear layers
    dtype=torch.qint8     # 8-bit integer
)

# Save quantized model
quant_path = '/content/LightweightCNN_quantized.pth'
torch.save(quantized_model.state_dict(), quant_path)

# Compare sizes
original_size = os.path.getsize(MODELS_DIR/'LightweightCNN_best.pth') / (1024**2)
quantized_size = os.path.getsize(quant_path) / (1024**2)

print(f'\nOriginal LightweightCNN:  {original_size:.2f} MB (float32)')
print(f'Quantized LightweightCNN: {quantized_size:.2f} MB (int8)')
print(f'Size reduction:           {original_size/quantized_size:.1f}x smaller')
print('\nQuantization complete.')

In [ ]:
# CELL 10 — Measure CPU inference time (FIXED)
print('Measuring CPU inference times...')
print('(Simulating smartphone deployment conditions)\n')

# Load a small batch for timing
X_sample = np.load(DRIVE_DIR/'X_eval.npy', mmap_mode='r')[:200].copy()
X_tensor  = torch.FloatTensor(X_sample).unsqueeze(1)  # (200, 1, 64, 128)

def time_model_on_cpu(model, X_tensor, n_runs=5):
    model.eval()
    times = []
    with torch.no_grad():
        for _ in range(n_runs):
            t = time.time()
            model(X_tensor)
            times.append((time.time()-t) / len(X_tensor) * 1000)
    return round(np.mean(times), 2)


# Load each model correctly
m_light = LightweightCNN()
m_light.load_state_dict(torch.load(MODELS_DIR/'LightweightCNN_best.pth', map_location='cpu'))

m_full = FullCNN()
m_full.load_state_dict(torch.load(MODELS_DIR/'FullCNN_best.pth', map_location='cpu'))

m_quant = quantized_model.cpu()

cpu_results = {
    'LightweightCNN (float32)': time_model_on_cpu(m_light,  X_tensor),
    'LightweightCNN (int8)':    time_model_on_cpu(m_quant,  X_tensor),
    'FullCNN (float32)':        time_model_on_cpu(m_full,   X_tensor),
}

for name, ms in cpu_results.items():
    status = 'REAL-TIME VIABLE' if ms < 50 else 'TOO SLOW FOR MOBILE'
    print(f'  {name:35s}: {ms:.2f} ms/sample  [{status}]')

print('\nCPU inference measurement complete.')

In [ ]:
# CELL 11 — Evaluate quantized model on all conditions
# Confirms the quantized model maintains accuracy after size reduction.

print('=== EVALUATING QUANTIZED LIGHTWEIGHT CNN ===')
quant_results = []

quantized_model_eval = quantized_model.cpu()
quantized_model_eval.eval()

for cond_name, distortion_fn in conditions:
    X_mmap = np.load(DRIVE_DIR/'X_eval.npy', mmap_mode='r')
    all_preds, all_scores = [], []
    start = time.time()
    CHUNK = 500

    for i in range(0, len(y_eval), CHUNK):
        chunk = X_mmap[i:i+CHUNK].copy()
        if distortion_fn:
            chunk = np.array([distortion_fn(s) for s in chunk])
        X_t = torch.FloatTensor(chunk).unsqueeze(1)
        with torch.no_grad():
            out = quantized_model_eval(X_t).squeeze(1)
        all_preds.extend((out >= 0.5).long().numpy())
        all_scores.extend(out.numpy())
        del chunk, X_t

    inf_ms = (time.time()-start)/len(y_eval)*1000
    preds  = np.array(all_preds)
    scores = np.array(all_scores)
    acc    = accuracy_score(y_eval, preds)*100
    eer    = compute_eer(y_eval, scores)
    cm     = confusion_matrix(y_eval, preds)
    tn,fp,fn,tp = cm.ravel()
    far = fp/(fp+tn)*100; frr = fn/(fn+tp)*100
    size_mb = os.path.getsize(quant_path)/(1024**2)

    print(f'  [{cond_name}] Acc: {acc:.2f}%  EER: {eer:.2f}%  '
          f'FAR: {far:.2f}%  Size: {size_mb:.2f}MB  Inf: {inf_ms:.1f}ms')

    quant_results.append({
        'Model':'LightweightCNN-INT8','Condition':cond_name,
        'Accuracy (%)':round(acc,2),'EER (%)':eer,
        'FAR (%)':round(far,2),'FRR (%)':round(frr,2),
        'Size (MB)':round(size_mb,2),'Inference (ms)':round(inf_ms,1)
    })
    del X_mmap; gc.collect()

print('\nQuantized model evaluation complete.')

In [ ]:
# CELL 12 — Generate complete IEEE results tables
# Table 2: All models under all conditions
# This is the main results section of your IEEE paper.

all_results = gmm_all_results + light_all_results + full_all_results + quant_results
results_df  = pd.DataFrame(all_results)

print('\n' + '='*90)
print('TABLE 2: COMPLETE RESULTS — ALL MODELS UNDER CLEAN AND DISTORTED CONDITIONS')
print('='*90)
print(results_df.to_string(index=False))
print('='*90)

# Pivot for cleaner IEEE table format
pivot = results_df.pivot_table(
    index='Model',
    columns='Condition',
    values='Accuracy (%)',
    aggfunc='first'
)[['Clean','G711','Noise','Combined']]

print('\n--- ACCURACY SUMMARY TABLE (for IEEE paper) ---')
print(pivot.to_string())

# Key findings
light_clean    = results_df[(results_df.Model=='LightweightCNN') & (results_df.Condition=='Clean')]['Accuracy (%)'].values[0]
light_combined = results_df[(results_df.Model=='LightweightCNN') & (results_df.Condition=='Combined')]['Accuracy (%)'].values[0]
quant_clean    = results_df[(results_df.Model=='LightweightCNN-INT8') & (results_df.Condition=='Clean')]['Accuracy (%)'].values[0]

print(f'\nKEY FINDINGS FOR IEEE PAPER:')
print(f'  1. LightweightCNN accuracy drop under combined distortion: {light_clean-light_combined:.2f}%')
print(f'     → This is your domain shift finding (Document 2, Section 4.1)')
print(f'  2. Quantization accuracy loss: {light_clean-quant_clean:.2f}%')
print(f'     → Near-zero loss confirms INT8 quantization is viable for mobile')

# Save
results_df.to_csv('/content/results_table2_all_conditions.csv', index=False)
pivot.to_csv('/content/results_accuracy_pivot.csv')
print('\nResults saved.')

In [ ]:
# CELL 13 — Plot accuracy degradation under distortion
# Bar chart showing how each model's accuracy drops under distortion.
# For IEEE paper: this is your key results figure.

models_to_plot = ['GMM-Baseline', 'LightweightCNN', 'FullCNN', 'LightweightCNN-INT8']
cond_labels    = ['Clean', 'G711', 'Noise', 'Combined']
colors         = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(cond_labels))
width = 0.2

for i, (model_name, color) in enumerate(zip(models_to_plot, colors)):
    accs = []
    for cond in cond_labels:
        val = results_df[
            (results_df.Model==model_name) &
            (results_df.Condition==cond)
        ]['Accuracy (%)'].values
        accs.append(val[0] if len(val) > 0 else 0)
    bars = ax.bar(x + i*width, accs, width, label=model_name, color=color, alpha=0.85)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'{acc:.1f}', ha='center', va='bottom', fontsize=7, rotation=45)

ax.set_xlabel('Audio Condition', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Model Accuracy Under Clean and Distorted Telephone Conditions', fontsize=13)
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(cond_labels, fontsize=11)
ax.set_ylim([0, 105])
ax.legend(loc='lower left', fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=95, color='gray', linestyle='--', alpha=0.5, label='95% threshold')

plt.tight_layout()
plt.savefig('/content/accuracy_under_distortion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Accuracy degradation chart saved.')

In [ ]:
# CELL 14 — Model size and inference time comparison chart
# For IEEE paper: shows the mobile deployment trade-off clearly.

model_names = ['GMM\nBaseline', 'LightweightCNN\n(float32)', 'LightweightCNN\n(int8)', 'FullCNN\n(float32)']
sizes_mb    = [
    gmm_all_results[0]['Size (MB)'],
    light_all_results[0]['Size (MB)'],
    quant_results[0]['Size (MB)'],
    full_all_results[0]['Size (MB)']
]
inf_times = [
    gmm_all_results[0]['Inference (ms)'],
    cpu_results.get('LightweightCNN (float32)', 0),
    cpu_results.get('LightweightCNN (int8)', 0),
    cpu_results.get('FullCNN (float32)', 0)
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Size and CPU Inference Time Comparison', fontsize=13)

colors = ['#95a5a6', '#3498db', '#2ecc71', '#e74c3c']
bars1 = ax1.bar(model_names, sizes_mb, color=colors, edgecolor='black', linewidth=0.5)
ax1.set_ylabel('Model Size (MB)', fontsize=11)
ax1.set_title('Model Size')
ax1.axhline(y=2, color='red', linestyle='--', alpha=0.7, label='2MB mobile target')
ax1.legend()
for bar, val in zip(bars1, sizes_mb):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
             f'{val:.2f}MB', ha='center', fontsize=9)

bars2 = ax2.bar(model_names, inf_times, color=colors, edgecolor='black', linewidth=0.5)
ax2.set_ylabel('Inference Time (ms/sample)', fontsize=11)
ax2.set_title('CPU Inference Time (Mobile Simulation)')
ax2.axhline(y=50, color='red', linestyle='--', alpha=0.7, label='50ms real-time target')
ax2.legend()
for bar, val in zip(bars2, inf_times):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'{val:.1f}ms', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('/content/size_and_speed_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Size and speed chart saved.')

In [ ]:
# CELL 15 — Save everything to Google Drive
# Final save of all Week 3 outputs.

# Save quantized model
shutil.copy(quant_path, MODELS_DIR/'LightweightCNN_quantized.pth')

# Save charts
for chart in ['distortion_comparison.png',
              'accuracy_under_distortion.png',
              'size_and_speed_comparison.png']:
    shutil.copy(f'/content/{chart}', CHARTS_DIR/chart)

# Save results
shutil.copy('/content/results_table2_all_conditions.csv',
            RESULTS_DIR/'results_table2_all_conditions.csv')
shutil.copy('/content/results_accuracy_pivot.csv',
            RESULTS_DIR/'results_accuracy_pivot.csv')

print('All Week 3 outputs saved to Google Drive.')
print('\n=== WEEK 3 COMPLETE ===')
print('\nYou now have everything needed for the IEEE paper:')
print('  CHARTS:  class_distribution, spectrogram_comparison,')
print('           distortion_comparison, training_curves,')
print('           confusion_matrices, accuracy_under_distortion,')
print('           size_and_speed_comparison')
print('  RESULTS: Table 1 (clean audio), Table 2 (all conditions)')
print('  MODELS:  GMM, LightweightCNN, FullCNN, LightweightCNN-INT8')
print('\nNext step: Week 4 — Write the IEEE paper')

In [ ]:
# Run this in your existing Colab notebook
# Make sure light_history and full_history are still in memory
# If session reset, you need to retrain first (run Week 2 cells 1-9)

import matplotlib.pyplot as plt
from pathlib import Path
import shutil

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('CNN Training Curves', fontsize=14)

for col, (history, name) in enumerate([
    (light_history, 'LightweightCNN'),
    (full_history,  'FullCNN')
]):
    ep = range(1, len(history['train_loss'])+1)

    # Loss plot
    axes[0][col].plot(ep, history['train_loss'], 'b-o', ms=3, label='Train')
    axes[0][col].plot(ep, history['val_loss'],   'r-o', ms=3, label='Val')
    axes[0][col].set_xlabel('Epoch')
    axes[0][col].set_ylabel('Loss')
    axes[0][col].set_title(f'{name} — Loss')
    axes[0][col].legend()
    axes[0][col].grid(alpha=0.3)

    # Accuracy plot
    axes[1][col].plot(ep, history['train_acc'], 'b-o', ms=3, label='Train')
    axes[1][col].plot(ep, history['val_acc'],   'r-o', ms=3, label='Val')
    axes[1][col].set_xlabel('Epoch')
    axes[1][col].set_ylabel('Accuracy (%)')
    axes[1][col].set_title(f'{name} — Accuracy (%)')
    axes[1][col].legend()
    axes[1][col].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves_v2.png', dpi=150, bbox_inches='tight')
plt.show()

# Save to Drive
CHARTS_DIR = Path('/content/drive/MyDrive/ICAST_Project/charts')
shutil.copy('/content/training_curves_v2.png', CHARTS_DIR/'training_curves.png')
print('Updated training curves saved to Drive.')

In [ ]:
# SNR vs ACCURACY EXPERIMENT
# Tests all four models across 6 SNR levels
# Expected time: 20-30 minutes total

import numpy as np
import torch
import pickle
import os
import time
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# ── Paths ──────────────────────────────────────────────────
DRIVE_DIR   = Path('/content/drive/MyDrive/ICAST_Project/features')
MODELS_DIR  = Path('/content/drive/MyDrive/ICAST_Project/models')
CHARTS_DIR  = Path('/content/drive/MyDrive/ICAST_Project/charts')
RESULTS_DIR = Path('/content/drive/MyDrive/ICAST_Project/results')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── SNR levels to test ─────────────────────────────────────
SNR_LEVELS = [0, 5, 10, 15, 20, 25, 30]
print(f'SNR levels to test: {SNR_LEVELS} dB')

In [ ]:
# ── Model definitions ──────────────────────────────────────
import torch.nn as nn

class LightweightCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.BatchNorm2d(16),
            nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32),
            nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64),
            nn.ReLU(), nn.MaxPool2d(2,2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(64*8*16, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.classifier(self.features(x))


class FullCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32),
            nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64),
            nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128),
            nn.ReLU(), nn.MaxPool2d(2,2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(128*8*16, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.classifier(self.features(x))



# ── Load trained weights (CORRECTED) ──────────────────────

# Lightweight CNN
light_model = LightweightCNN()
light_model.load_state_dict(
    torch.load(MODELS_DIR/'LightweightCNN_best.pth', map_location=device))
light_model = light_model.to(device).eval()
print('Lightweight CNN loaded.')

# Full CNN
full_model = FullCNN()
full_model.load_state_dict(
    torch.load(MODELS_DIR/'FullCNN_best.pth', map_location=device))
full_model = full_model.to(device).eval()
print('Full CNN loaded.')

# Quantized CNN — load weights first, THEN quantize
light_for_quant = LightweightCNN()
light_for_quant.load_state_dict(
    torch.load(MODELS_DIR/'LightweightCNN_best.pth', map_location='cpu'))
light_for_quant.eval()
quant_model = torch.quantization.quantize_dynamic(
    light_for_quant, {nn.Linear}, dtype=torch.qint8)
quant_model = quant_model.cpu().eval()
print('Quantized model loaded.')

# GMM
with open(MODELS_DIR/'gmm_model.pkl', 'rb') as f:
    gmm_data = pickle.load(f)
gmm_real = gmm_data['gmm_real']
gmm_fake = gmm_data['gmm_fake']
scaler   = gmm_data['scaler']
print('GMM loaded.')

# Labels
y_eval = np.load(DRIVE_DIR/'y_eval.npy')
print('\nAll models loaded successfully. Ready to run SNR experiment.')

In [ ]:
# ── AWGN noise function at variable SNR ─────
def add_awgn_noise(spectrogram, snr_db):
    """Add Gaussian noise at a specific SNR level in dB."""
    signal_power = np.mean(spectrogram ** 2)
    noise_power  = signal_power / (10 ** (snr_db / 10))
    noise = np.random.normal(0, np.sqrt(noise_power), spectrogram.shape)
    return np.clip(spectrogram + noise, 0, 1)


# ── Dataset class ─────
class NoisyDataset(torch.utils.data.Dataset):
    def __init__(self, X_path, y_path, snr_db=None):
        self.X = np.load(X_path, mmap_mode='r')
        self.y = np.load(y_path)
        self.snr_db = snr_db

    def __len__(self): return len(self.y)

    def __getitem__(self, idx):
        spec = self.X[idx].copy()
        if self.snr_db is not None:
            spec = add_awgn_noise(spec, self.snr_db)
        x = torch.FloatTensor(spec).unsqueeze(0)
        y = torch.FloatTensor([self.y[idx]])
        return x, y


# ── Evaluation functions ──────────
def evaluate_cnn_snr(model, snr_db, use_cpu=False):
    dataset = NoisyDataset(
        DRIVE_DIR/'X_eval.npy',
        DRIVE_DIR/'y_eval.npy',
        snr_db=snr_db
    )
    loader = DataLoader(dataset, batch_size=64,
                       shuffle=False, num_workers=2)
    target_device = 'cpu' if use_cpu else device
    model = model.to(target_device).eval()
    all_preds = []
    with torch.no_grad():
        for X_b, _ in loader:
            out = model(X_b.to(target_device)).squeeze(1)
            all_preds.extend((out >= 0.5).long().cpu().numpy())
    return accuracy_score(y_eval, all_preds) * 100


def evaluate_gmm_snr(snr_db):
    X_mmap = np.load(DRIVE_DIR/'X_eval.npy', mmap_mode='r')
    CHUNK = 1000
    all_preds = []
    for i in range(0, len(y_eval), CHUNK):
        chunk = X_mmap[i:i+CHUNK].copy()
        if snr_db is not None:
            chunk = np.array([add_awgn_noise(s, snr_db) for s in chunk])
        chunk_flat = chunk.reshape(len(chunk), -1)
        chunk_s    = scaler.transform(chunk_flat)
        s_real = gmm_real.score_samples(chunk_s)
        s_fake = gmm_fake.score_samples(chunk_s)
        all_preds.extend((s_fake > s_real).astype(int))
    return accuracy_score(y_eval, all_preds) * 100


# ── Run the full SNR experiment ────────────────────────────
print('Running SNR vs Accuracy experiment...')
print('This will take approximately 20-30 minutes.\n')

results = {
    'GMM Baseline':          [],
    'Lightweight CNN':       [],
    'Full CNN':              [],
    'Lightweight CNN-INT8':  [],
}

# Add clean baseline (no noise) as reference point
for snr in SNR_LEVELS:
    print(f'\nSNR = {snr} dB')

    acc = evaluate_gmm_snr(snr)
    results['GMM Baseline'].append(acc)
    print(f'  GMM Baseline:         {acc:.2f}%')

    acc = evaluate_cnn_snr(light_model, snr)
    results['Lightweight CNN'].append(acc)
    print(f'  Lightweight CNN:      {acc:.2f}%')

    acc = evaluate_cnn_snr(full_model, snr)
    results['Full CNN'].append(acc)
    print(f'  Full CNN:             {acc:.2f}%')

    acc = evaluate_cnn_snr(quant_model, snr, use_cpu=True)
    results['Lightweight CNN-INT8'].append(acc)
    print(f'  Lightweight CNN-INT8: {acc:.2f}%')

print('\nExperiment complete.')
print('\nResults summary:')
print(f'{"SNR (dB)":<12}', end='')
for model_name in results:
    print(f'{model_name:<22}', end='')
print()
for i, snr in enumerate(SNR_LEVELS):
    print(f'{snr:<12}', end='')
    for model_name in results:
        print(f'{results[model_name][i]:<22.2f}', end='')
    print()

In [ ]:
# SNR vs ACCURACY PLOT — FIXED VERSION
# Lightweight CNN now clearly visible and distinguished

import matplotlib.pyplot as plt
import shutil
import numpy as np

fig, ax = plt.subplots(figsize=(12, 7))

# Use different line styles AND colors to distinguish overlapping lines
styles = {
    'GMM Baseline':         {'color':'#64748B', 'linestyle':'--',  'marker':'s', 'linewidth':2.0, 'markersize':8,  'zorder':3},
    'Lightweight CNN':      {'color':'#FF6B00', 'linestyle':'-',   'marker':'o', 'linewidth':3.0, 'markersize':10, 'zorder':5},
    'Full CNN':             {'color':'#EF4444', 'linestyle':'-.',  'marker':'^', 'linewidth':2.0, 'markersize':8,  'zorder':3},
    'Lightweight CNN-INT8': {'color':'#10B981', 'linestyle':':',   'marker':'D', 'linewidth':2.0, 'markersize':8,  'zorder':4},
}

# Vertical offset for annotations to avoid overlap
offsets = {
    'GMM Baseline':         -14,
    'Lightweight CNN':        8,
    'Full CNN':              -14,
    'Lightweight CNN-INT8':   8,
}

for model_name, accs in results.items():
    st = styles[model_name]
    ax.plot(SNR_LEVELS, accs,
            color=st['color'],
            linestyle=st['linestyle'],
            marker=st['marker'],
            linewidth=st['linewidth'],
            markersize=st['markersize'],
            zorder=st['zorder'],
            label=model_name)

    for snr, acc in zip(SNR_LEVELS, accs):
        ax.annotate(f'{acc:.1f}',
                   xy=(snr, acc),
                   xytext=(0, offsets[model_name]),
                   textcoords='offset points',
                   ha='center', fontsize=8,
                   color=st['color'],
                   fontweight='bold')

ax.set_xlabel('SNR (dB)', fontsize=13)
ax.set_ylabel('Accuracy (%)', fontsize=13)
ax.set_title('Recognition Accuracy vs SNR Level\nUnder AWGN Channel Conditions',
             fontsize=14, fontweight='bold')
ax.set_xticks(SNR_LEVELS)
ax.set_ylim([35, 110])
ax.legend(loc='lower right', fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3)

# 80% threshold line
ax.axhline(y=80, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.text(30.3, 80.5, '80% threshold', fontsize=8, color='gray')

# # Crossover annotation
# ax.annotate('Full CNN crossover\n~25-30 dB',
#             xy=(25, 84.59),
#             xytext=(18, 65),
#             fontsize=9, color='black',
#             arrowprops=dict(arrowstyle='->', color='black', lw=1.2))

# # Collapse annotation
# ax.annotate('Full CNN collapses\nat 0 dB (42%)',
#             xy=(0, 42.02),
#             xytext=(3, 50),
#             fontsize=9, color='#EF4444',
#             arrowprops=dict(arrowstyle='->', color='#EF4444', lw=1.2))

plt.tight_layout()
plt.savefig('/content/snr_vs_accuracy_v2.png', dpi=150, bbox_inches='tight')
plt.show()

shutil.copy('/content/snr_vs_accuracy_v2.png',
            CHARTS_DIR/'snr_vs_accuracy.png')
print('Updated SNR chart saved to Drive.')

In [ ]:
# COMPLETE RA / FAR / MAR FOR ALL MODELS ACROSS ALL CONDITIONS

import pandas as pd
from sklearn.metrics import confusion_matrix, roc_curve
import gc

def compute_eer(y_true, y_scores):
    fpr, tpr, _ = roc_curve(y_true, y_scores, pos_label=1)
    fnr = 1 - tpr
    idx = np.argmin(np.abs(fpr - fnr))
    return round((fpr[idx] + fnr[idx]) / 2 * 100, 2)


def get_full_metrics(y_true, y_pred, y_scores, model_name, condition):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    # RA  — Recognition Accuracy: % of fake voices correctly detected
    # FAR — False Alarm Rate: % of real voices wrongly flagged as fake
    # MAR — Miss Alarm Rate: % of fake voices wrongly accepted as real
    ra  = tp / (tp + fn) * 100   # True Positive Rate (sensitivity)
    far = fp / (fp + tn) * 100   # False Positive Rate
    mar = fn / (fn + tp) * 100   # False Negative Rate (miss rate)
    acc = (tp + tn) / (tp + tn + fp + fn) * 100
    eer = compute_eer(y_true, y_scores)

    return {
        'Model':     model_name,
        'Condition': condition,
        'Acc (%)':   round(acc, 2),
        'RA (%)':    round(ra,  2),
        'FAR (%)':   round(far, 2),
        'MAR (%)':   round(mar, 2),
        'EER (%)':   eer,
    }


# Distortion functions
def simulate_g711(spec):
    d = spec.copy()
    cutoff = int(d.shape[0] * 0.6)
    d[cutoff:, :] = 0.0
    d = np.clip(d + np.random.uniform(-0.02, 0.02, d.shape), 0, 1)
    return d

def add_noise_20db(spec):
    sp = np.mean(spec ** 2)
    np_ = sp / (10 ** (20 / 10))
    return np.clip(
        spec + np.random.normal(0, np.sqrt(np_), spec.shape), 0, 1)

def combined(spec):
    return add_noise_20db(simulate_g711(spec))

CONDITIONS = [
    ('Clean',    None),
    ('G.711',    simulate_g711),
    ('Noise',    add_noise_20db),
    ('Combined', combined),
]


# CNN evaluation
def eval_cnn_full(model, condition_fn, model_name,
                  condition, use_cpu=False):
    class CondDataset(torch.utils.data.Dataset):
        def __init__(self):
            self.X = np.load(DRIVE_DIR/'X_eval.npy', mmap_mode='r')
            self.y = np.load(DRIVE_DIR/'y_eval.npy')
        def __len__(self): return len(self.y)
        def __getitem__(self, idx):
            s = self.X[idx].copy()
            if condition_fn: s = condition_fn(s)
            return (torch.FloatTensor(s).unsqueeze(0),
                    torch.FloatTensor([self.y[idx]]))

    loader = torch.utils.data.DataLoader(
        CondDataset(), batch_size=64,
        shuffle=False, num_workers=2)
    tgt = 'cpu' if use_cpu else device
    model = model.to(tgt).eval()
    preds, scores = [], []
    with torch.no_grad():
        for X_b, _ in loader:
            out = model(X_b.to(tgt)).squeeze(1)
            preds.extend((out >= 0.5).long().cpu().numpy())
            scores.extend(out.cpu().numpy())
    del loader; gc.collect()
    return get_full_metrics(
        y_eval, np.array(preds), np.array(scores),
        model_name, condition)


# GMM evaluation
def eval_gmm_full(condition_fn, condition):
    X_mmap = np.load(DRIVE_DIR/'X_eval.npy', mmap_mode='r')
    CHUNK = 1000
    preds, scores = [], []
    for i in range(0, len(y_eval), CHUNK):
        chunk = X_mmap[i:i+CHUNK].copy()
        if condition_fn:
            chunk = np.array([condition_fn(s) for s in chunk])
        flat   = chunk.reshape(len(chunk), -1)
        flat_s = scaler.transform(flat)
        sr = gmm_real.score_samples(flat_s)
        sf = gmm_fake.score_samples(flat_s)
        preds.extend((sf > sr).astype(int))
        scores.extend(sf - sr)
        del chunk, flat, flat_s
    scores = np.array(scores)
    scores = (scores - scores.min()) / (
               scores.max() - scores.min() + 1e-8)
    del X_mmap; gc.collect()
    return get_full_metrics(
        y_eval, np.array(preds), scores,
        'GMM Baseline', condition)


# Run all evaluations
print('Computing RA / FAR / MAR for all models and conditions...\n')
all_metrics = []

for cond_name, cond_fn in CONDITIONS:
    print(f'Condition: {cond_name}')

    all_metrics.append(eval_gmm_full(cond_fn, cond_name))
    print(f'  GMM Baseline done')

    all_metrics.append(eval_cnn_full(
        light_model, cond_fn, 'Lightweight CNN', cond_name))
    print(f'  Lightweight CNN done')

    all_metrics.append(eval_cnn_full(
        full_model, cond_fn, 'Full CNN', cond_name))
    print(f'  Full CNN done')

    all_metrics.append(eval_cnn_full(
        quant_model, cond_fn,
        'Lightweight CNN-INT8', cond_name, use_cpu=True))
    print(f'  Lightweight CNN-INT8 done')


# Print results
df = pd.DataFrame(all_metrics)
print('\n' + '='*75)
print('TABLE: RA / FAR / MAR — ALL MODELS, ALL CONDITIONS')
print('='*75)
print(df.to_string(index=False))
print('='*75)
print('\nDefinitions:')
print('  RA  = Recognition Accuracy: % of fake voices correctly detected')
print('  FAR = False Alarm Rate:     % of real voices wrongly flagged as fake')
print('  MAR = Miss Alarm Rate:      % of fake voices wrongly accepted as real')

# Save
df.to_csv('/content/ra_far_mar_complete.csv', index=False)
shutil.copy('/content/ra_far_mar_complete.csv',
            RESULTS_DIR/'ra_far_mar_complete.csv')
print('\nResults saved to Drive.')

In [ ]:
# IDENTIFICATION LATENCY — END-TO-END PIPELINE TIMING
# Measures total time from raw spectrogram to decision
# Includes: feature loading + optional distortion + model inference
# This is the real deployment latency a mobile banking app would experience

import time
import numpy as np
import torch

# Load a batch of 200 samples for timing
X_sample = np.load(DRIVE_DIR/'X_eval.npy', mmap_mode='r')[:200].copy()
y_sample  = np.load(DRIVE_DIR/'y_eval.npy')[:200]

N_RUNS   = 10   # average over 10 runs to reduce variance
N_SAMPLE = 200  # samples per run

print('Measuring end-to-end identification latency...')
print(f'Averaging over {N_RUNS} runs on {N_SAMPLE} samples each.\n')


def time_end_to_end_cnn(model, X_sample, n_runs=10, use_cpu=False):
    """
    Times the full pipeline:
    Step 1 — Convert numpy array to tensor (simulates feature loading)
    Step 2 — Apply model inference
    Returns average latency per sample in milliseconds.
    """
    tgt = 'cpu' if use_cpu else device
    model = model.to(tgt).eval()
    times = []

    for _ in range(n_runs):
        start = time.perf_counter()
        # Step 1: feature preparation
        X_tensor = torch.FloatTensor(X_sample).unsqueeze(1).to(tgt)
        # Step 2: inference
        with torch.no_grad():
            _ = model(X_tensor)
        end = time.perf_counter()
        elapsed_ms = (end - start) / len(X_sample) * 1000
        times.append(elapsed_ms)

    return round(np.mean(times), 3), round(np.std(times), 3)


def time_end_to_end_gmm(X_sample, n_runs=10):
    """
    Times the full GMM pipeline:
    Step 1 — Flatten spectrogram
    Step 2 — Standardise with scaler
    Step 3 — Score against both GMMs and decide
    """
    times = []

    for _ in range(n_runs):
        start = time.perf_counter()
        # Step 1: flatten
        X_flat = X_sample.reshape(len(X_sample), -1)
        # Step 2: standardise
        X_scaled = scaler.transform(X_flat)
        # Step 3: score and decide
        s_real = gmm_real.score_samples(X_scaled)
        s_fake = gmm_fake.score_samples(X_scaled)
        _      = (s_fake > s_real).astype(int)
        end = time.perf_counter()
        elapsed_ms = (end - start) / len(X_sample) * 1000
        times.append(elapsed_ms)

    return round(np.mean(times), 3), round(np.std(times), 3)


# Run timing for all models
print('─' * 60)
latency_results = []

# GMM Baseline
mean_t, std_t = time_end_to_end_gmm(X_sample)
print(f'GMM Baseline:          {mean_t:.3f} ms ± {std_t:.3f} ms per sample')
latency_results.append({
    'Model': 'GMM Baseline',
    'Mean Latency (ms)': mean_t,
    'Std Dev (ms)': std_t,
    'Device': 'CPU',
    'Real-Time Viable': 'Yes' if mean_t < 50 else 'No'
})

# Lightweight CNN — GPU
mean_t, std_t = time_end_to_end_cnn(light_model, X_sample)
print(f'Lightweight CNN (GPU): {mean_t:.3f} ms ± {std_t:.3f} ms per sample')
latency_results.append({
    'Model': 'Lightweight CNN (GPU)',
    'Mean Latency (ms)': mean_t,
    'Std Dev (ms)': std_t,
    'Device': 'GPU',
    'Real-Time Viable': 'Yes' if mean_t < 50 else 'No'
})

# Lightweight CNN — CPU (simulates smartphone)
mean_t, std_t = time_end_to_end_cnn(light_model, X_sample, use_cpu=True)
print(f'Lightweight CNN (CPU): {mean_t:.3f} ms ± {std_t:.3f} ms per sample')
latency_results.append({
    'Model': 'Lightweight CNN (CPU)',
    'Mean Latency (ms)': mean_t,
    'Std Dev (ms)': std_t,
    'Device': 'CPU',
    'Real-Time Viable': 'Yes' if mean_t < 50 else 'No'
})

# Lightweight CNN-INT8 — CPU
mean_t, std_t = time_end_to_end_cnn(quant_model, X_sample, use_cpu=True)
print(f'Lightweight INT8 (CPU):{mean_t:.3f} ms ± {std_t:.3f} ms per sample')
latency_results.append({
    'Model': 'Lightweight CNN-INT8 (CPU)',
    'Mean Latency (ms)': mean_t,
    'Std Dev (ms)': std_t,
    'Device': 'CPU',
    'Real-Time Viable': 'Yes' if mean_t < 50 else 'No'
})

# Full CNN — GPU
mean_t, std_t = time_end_to_end_cnn(full_model, X_sample)
print(f'Full CNN (GPU):        {mean_t:.3f} ms ± {std_t:.3f} ms per sample')
latency_results.append({
    'Model': 'Full CNN (GPU)',
    'Mean Latency (ms)': mean_t,
    'Std Dev (ms)': std_t,
    'Device': 'GPU',
    'Real-Time Viable': 'Yes' if mean_t < 50 else 'No'
})

# Full CNN — CPU
mean_t, std_t = time_end_to_end_cnn(full_model, X_sample, use_cpu=True)
print(f'Full CNN (CPU):        {mean_t:.3f} ms ± {std_t:.3f} ms per sample')
latency_results.append({
    'Model': 'Full CNN (CPU)',
    'Mean Latency (ms)': mean_t,
    'Std Dev (ms)': std_t,
    'Device': 'CPU',
    'Real-Time Viable': 'Yes' if mean_t < 50 else 'No'
})

print('─' * 60)

# Print summary table
import pandas as pd
df_latency = pd.DataFrame(latency_results)
print('\n=== IDENTIFICATION LATENCY SUMMARY ===')
print(df_latency.to_string(index=False))
print('\nNote: CPU measurements simulate smartphone deployment conditions.')
print('Real-time target: under 50 ms per sample.')

# Save
df_latency.to_csv('/content/identification_latency.csv', index=False)
shutil.copy('/content/identification_latency.csv',
            RESULTS_DIR/'identification_latency.csv')
print('\nLatency results saved to Drive.')

In [ ]:
# IDENTIFICATION LATENCY BAR CHART

fig, ax = plt.subplots(figsize=(11, 6))

models   = [
    'GMM\nBaseline\n(CPU)',
    'Lightweight\nCNN (GPU)',
    'Lightweight\nCNN (CPU)',
    'Lightweight\nCNN-INT8\n(CPU)',
    'Full CNN\n(GPU)',
    'Full CNN\n(CPU)'
]
means    = [0.313, 0.171, 4.962, 2.343, 0.107, 6.584]
stds     = [0.100, 0.056, 1.688, 0.096, 0.024, 0.970]
colors   = [
    '#64748B',   # GMM — grey
    '#1A56DB',   # Lightweight GPU — blue
    '#FF6B00',   # Lightweight CPU — orange
    '#10B981',   # INT8 CPU — green
    '#7C3AED',   # Full GPU — purple
    '#EF4444',   # Full CPU — red
]

bars = ax.bar(models, means, yerr=stds,
              color=colors, capsize=5,
              edgecolor='black', linewidth=0.5,
              error_kw={'linewidth': 1.5})

# Value labels on bars
for bar, mean, std in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + std + 0.1,
            f'{mean:.3f} ms',
            ha='center', va='bottom',
            fontsize=9, fontweight='bold')

# 50ms real-time target line
ax.axhline(y=50, color='red', linestyle='--',
           alpha=0.6, linewidth=1.5)
ax.text(5.5, 51, '50 ms\ntarget',
        fontsize=8, color='red', ha='center')

ax.set_ylabel('Mean Latency (ms/sample)', fontsize=12)
ax.set_title('End-to-End Identification Latency\nAll Models — CPU and GPU',
             fontsize=13, fontweight='bold')
ax.set_ylim([0, 12])
ax.grid(axis='y', alpha=0.3)

# Annotate CPU vs GPU
ax.text(0.02, 0.97,
        'Orange/green/grey bars = CPU (smartphone simulation)\n'
        'Blue/purple bars = GPU',
        transform=ax.transAxes,
        fontsize=8, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.4))

plt.tight_layout()
plt.savefig('/content/identification_latency.png',
            dpi=150, bbox_inches='tight')
plt.show()

shutil.copy('/content/identification_latency.png',
            CHARTS_DIR/'identification_latency.png')
print('Latency chart saved to Drive.')

In [ ]:
# FULL RECOVERY CELL — run this first
import numpy as np
import torch
import torch.nn as nn
import pickle
import shutil
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR   = Path('/content/drive/MyDrive/ICAST_Project/features')
MODELS_DIR  = Path('/content/drive/MyDrive/ICAST_Project/models')
CHARTS_DIR  = Path('/content/drive/MyDrive/ICAST_Project/charts')
RESULTS_DIR = Path('/content/drive/MyDrive/ICAST_Project/results')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class LightweightCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.BatchNorm2d(16),
            nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(16,32,3,padding=1), nn.BatchNorm2d(32),
            nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64),
            nn.ReLU(), nn.MaxPool2d(2,2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(64*8*16, 64), nn.ReLU(),
            nn.Linear(64, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.classifier(self.features(x))


class FullCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32),
            nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64),
            nn.ReLU(), nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128),
            nn.ReLU(), nn.MaxPool2d(2,2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(128*8*16, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 1), nn.Sigmoid()
        )
    def forward(self, x): return self.classifier(self.features(x))


print(f'Device: {device}')
print('All classes and paths defined.')
print('Ready to run parameter count cell.')

In [ ]:
# PARAMETER COUNT VERIFICATION
light = LightweightCNN()
light.load_state_dict(
    torch.load(MODELS_DIR/'LightweightCNN_best.pth', map_location='cpu'))

full = FullCNN()
full.load_state_dict(
    torch.load(MODELS_DIR/'FullCNN_best.pth', map_location='cpu'))

light_params = sum(p.numel() for p in light.parameters())
full_params  = sum(p.numel() for p in full.parameters())

print(f'LightweightCNN parameters: {light_params:,}')
print(f'FullCNN parameters:        {full_params:,}')
print(f'Ratio:                     {full_params/light_params:.1f}x')
print(f'\nLightweight theoretical:   {light_params*4/1e6:.2f} MB (float32)')
print(f'Full theoretical:          {full_params*4/1e6:.2f} MB (float32)')
print(f'\nLightweight saved file:    '
      f'{(MODELS_DIR/"LightweightCNN_best.pth").stat().st_size/(1024**2):.2f} MB')
print(f'Full saved file:           '
      f'{(MODELS_DIR/"FullCNN_best.pth").stat().st_size/(1024**2):.2f} MB')

In [ ]:
# Check if raw audio files are still available
from pathlib import Path

# Check Colab local storage first
local = Path('/content/asvspoof2019/LA')
if local.exists():
    print('Audio found in local Colab storage.')
    for folder in sorted(local.iterdir()):
        print(f'  {folder.name}/')
else:
    print('Audio NOT in local storage — session was reset.')

# Check Drive
drive_audio = Path('/content/drive/MyDrive/ICAST_Project')
print(f'\nContents of ICAST_Project on Drive:')
for f in sorted(drive_audio.iterdir()):
    print(f'  {f.name}/')

In [ ]:
# DIRECT DOWNLOAD — bypasses Kaggle entirely
#This is an alternative way to download the ASVspoof 2019 dataset
import os
from pathlib import Path

os.makedirs('/content/asvspoof2019', exist_ok=True)

print('Downloading ASVspoof 2019 LA partition...')
print('Expected size: ~8 GB. Expected time: 15-25 minutes.')
print('Do not close the tab.\n')

# Download only the LA partition — we do not need PA
!wget --no-check-certificate \
    "https://datashare.ed.ac.uk/bitstream/handle/10283/3336/LA.zip" \
    -O /content/asvspoof2019/LA.zip \
    --progress=bar:force 2>&1

print('\nChecking download:')
!ls -lh /content/asvspoof2019/

In [ ]:
# UNZIP THE DOWNLOADED FILE
print('Unzipping LA.zip — this takes about 5 minutes...')
print('Do not close the tab.\n')

!unzip -q /content/asvspoof2019/LA.zip -d /content/asvspoof2019/

print('Unzip complete. Checking contents:')
!ls -lh /content/asvspoof2019/

print('\nChecking LA folder:')
!ls /content/asvspoof2019/LA/

In [ ]:
# CONFIRM AUDIO FILES ARE ACCESSIBLE
from pathlib import Path
import pandas as pd

LA_DIR = Path('/content/asvspoof2019/LA')

# Check folder structure
print('LA directory contents:')
for folder in sorted(LA_DIR.iterdir()):
    print(f'  {folder.name}/')

# Check protocol files
PROTO_DIR = LA_DIR / 'ASVspoof2019_LA_cm_protocols'
eval_proto = PROTO_DIR / 'ASVspoof2019.LA.cm.eval.trl.txt'
COL_NAMES  = ['speaker_id', 'filename', 'environment', 'attack', 'label']
df_eval    = pd.read_csv(eval_proto, sep=' ', header=None, names=COL_NAMES)

print(f'\nEvaluation protocol loaded: {len(df_eval):,} files')
print(f'  Real (bonafide): {(df_eval.label=="bonafide").sum():,}')
print(f'  Fake (spoof):    {(df_eval.label=="spoof").sum():,}')

# Check one audio file exists
EVAL_AUDIO = LA_DIR / 'ASVspoof2019_LA_eval' / 'flac'
sample_file = EVAL_AUDIO / f'{df_eval.filename.iloc[0]}.flac'
print(f'\nSample audio file exists: {sample_file.exists()}')
print(f'Sample path: {sample_file}')

# Count attack types
print(f'\nAttack types in evaluation set:')
print(df_eval[df_eval.label=="spoof"].attack.value_counts().to_string())